# FMCG lakehouse: querying GCS through Hive with PySpark

Run this notebook with the **FMCG PySpark (Spark 4.0.1, GCS + Hive)** kernel. When the kernel starts, it creates:
- `spark`: a SparkSession with the Google Cloud Storage connector and the Hive metastore (the same settings as the pipeline jobs);
- `F`: `pyspark.sql.functions`.

The data covers **three years of daily drops (14 Nov 2023 to 13 Nov 2026)**. Three Hive databases are available:

| Database | Bucket | Partitions |
|---|---|---|
| `datalake` | `gs://fmcg-datalake` | `year`, `month`, `day` |
| `mart` | `gs://fmcg-mart` | `year`, `week` (ISO) |
| `qa` | `gs://fmcg-mart/_qa` | `run_id` (QA check results) |

Shell cells such as `!gsutil ls -lh gs://fmcg-datalake/` browse the buckets (section 8).

Always filter on the partition columns: Spark then reads only those GCS folders.

In [ ]:
spark.version, spark.catalog.currentDatabase()

## 0. Configure the notebook for GCS (`gsutil`, `gcloud storage`, Spark)

This cell points the Google Cloud CLI at the project's GCS: the local emulator by default, or real GCS when `GCS_MODE=real`.
- It creates (once) a separate gcloud configuration called `fmcg-emulator` and selects it **only for this notebook**, so your default gcloud setup is not changed.
- It makes `gsutil` run through `gcloud storage`, which supports the emulator endpoint.
- It reuses the `spark` session the kernel started; if this notebook runs on another kernel, it creates one.

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO = Path(os.environ.get("FMCG_REPO", Path.cwd().parent)).resolve()
sys.path.insert(0, str(REPO))
GCS_MODE = os.environ.setdefault("GCS_MODE", "emulator")

if GCS_MODE == "emulator":
    # one-time: named gcloud configuration "fmcg-emulator" -> http://localhost:4443 (no login needed)
    subprocess.run([str(REPO / "scripts" / "setup_gcloud_emulator.sh")], check=True, capture_output=True)
    os.environ["CLOUDSDK_ACTIVE_CONFIG_NAME"] = "fmcg-emulator"        # only this notebook's shell commands use it
    os.environ["BOTO_CONFIG"] = str(REPO / ".gcloud" / "boto.cfg")      # gsutil -> gcloud storage (emulator-aware)
else:
    os.environ.pop("CLOUDSDK_ACTIVE_CONFIG_NAME", None)                 # real GCS: your normal gcloud login/project
os.environ["CLOUDSDK_COMPONENT_MANAGER_DISABLE_UPDATE_CHECK"] = "1"     # no "updates available" banner in outputs

if "spark" not in globals():                                           # kernel already created it; other kernels: create now
    from common.config import get_spark
    from pyspark.sql import functions as F
    spark = get_spark("jupyter-fmcg")

print("GCS mode         :", GCS_MODE)
print("gcloud config    :", os.environ.get("CLOUDSDK_ACTIVE_CONFIG_NAME", "default"))
print("spark            :", spark.version)

In [ ]:
!gcloud storage ls
!gsutil ls -lh gs://fmcg-datalake/

## 1. What is in each layer (`spark.sql(...).show(10, 0)`)

In [ ]:
spark.sql("show databases").show(10, 0)
spark.sql("show tables in datalake").show(10, 0)
spark.sql("show tables in mart").show(10, 0)
spark.sql("show tables in qa").show(10, 0)

In [ ]:
spark.sql("show partitions mart.fact_sales").show(10, 0)

In [ ]:
# where a table's files live in GCS, and how it is stored
spark.sql("describe formatted mart.store_weekly_performance") \
     .where("col_name in ('Type', 'Provider', 'Location', 'Partition Provider')") \
     .show(10, 0)

## 2. Raw datalake with `spark.table(...)`

In [ ]:
tx = spark.table("datalake.transactions")
tx.printSchema()

In [ ]:
# one day's raw POS lines (partition pruning: only gs://fmcg-datalake/transactions/year=2026/month=11/day=8/ is read)
tx.where("year = 2026 and month = 11 and day = 8") \
  .select("transaction_id", "line_no", "transaction_ts", "store_id", "product_name", "quantity", "unit_price", "filename", "modified_dttm") \
  .show(10, 0)

In [ ]:
# lines and gross value per drop date, with the DataFrame API
# lines and gross value per month, with the DataFrame API (3 years of daily partitions)
(tx.groupBy("year", "month")
   .agg(F.countDistinct("day").alias("days"), F.count("*").alias("lines"),
        F.round(F.sum(F.col("quantity") * F.col("unit_price")), 0).alias("gross_inr"))
   .orderBy("year", "month")
   .show(40, 0))

## 3. Business questions on the mart with `spark.sql(...)`

In [ ]:
spark.sql("""
    select week, region,
           round(sum(net_sales))           as net_sales_inr,
           round(sum(prev_week_net_sales)) as prev_week_inr,
           round(100 * (sum(net_sales) / sum(prev_week_net_sales) - 1), 1) as wow_pct
    from mart.store_weekly_performance
    where year = 2026 and week in (44, 45)
    group by week, region
    order by week, net_sales_inr desc
""").show(10, 0)

In [ ]:
# top stores in the Diwali week (ISO week 45), using spark.table + DataFrame API
(spark.table("mart.store_weekly_performance")
      .where("year = 2026 and week = 45")
      .select("store_name", "region", "store_format", "net_sales", "avg_basket_value", "sales_per_sqft", "wow_growth_pct", "rank_in_region")
      .orderBy(F.desc("net_sales"))
      .show(10, 0))

In [ ]:
spark.sql("""
    select category,
           round(sum(net_sales)) as net_sales_inr,
           round(100 * sum(net_sales) / sum(sum(net_sales)) over (), 1) as share_pct,
           round(100 * (sum(net_sales) / sum(prev_week_net_sales) - 1), 1) as wow_pct
    from mart.category_brand_weekly
    where year = 2026 and week = 45
    group by category
    order by net_sales_inr desc
""").show(10, 0)

In [ ]:
# did the promotions pay for their discount?
spark.sql("""
    select week, promo_name, brand, discount_pct, promo_units, unit_uplift_pct,
           discount_cost, incremental_net_sales, roi
    from mart.promotion_weekly_effectiveness
    where year = 2026 and week between 43 and 46
    order by week, roi desc
""").show(10, 0)

In [ ]:
spark.table("mart.customer_tier_weekly").where("year = 2026 and week = 45") \
     .orderBy(F.desc("net_sales")).show(10, 0)

## 4. Mixing both styles: a DataFrame registered as a temporary view

In [ ]:
daily = (spark.table("mart.fact_sales")
              .where("year = 2026 and week in (44, 45)")
              .groupBy("sale_date", "day_of_week")
              .agg(F.round(F.sum("net_amount"), 0).alias("net_sales_inr"),
                   F.countDistinct("transaction_id").alias("baskets")))
daily.createOrReplaceTempView("daily_sales")

spark.sql("""
    select sale_date, day_of_week, net_sales_inr, baskets,
           round(net_sales_inr / baskets, 0) as avg_basket_inr
    from daily_sales
    order by sale_date
""").show(14, 0)

In [ ]:
import matplotlib.pyplot as plt

pdf = spark.table("daily_sales").orderBy("sale_date").toPandas()
fig, ax = plt.subplots(figsize=(10, 3.5))
# Spark decimals arrive as Python Decimal objects: convert to float for plotting
ax.bar(pdf["sale_date"].astype(str), pdf["net_sales_inr"].astype(float) / 1e5, color="#0f6e78")
ax.set_ylabel("Net sales (₹ lakh)")
ax.set_title("Daily net sales, ISO weeks 44-45 of 2026 (Diwali on 8 Nov)")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

## 5. Proof of partition pruning

The physical plan lists `PartitionFilters`. Spark lists only the `year=2026/week=45` folder in GCS.

In [ ]:
spark.sql("select sum(net_amount) from mart.fact_sales where year = 2026 and week = 45").explain()

## 6. Three years of weekly KPIs: week-on-week and year-on-year

In [ ]:
spark.sql("""
    select year, week, week_start, week_end, stores_trading, net_sales,
           wow_growth_pct, yoy_growth_pct, online_share_pct
    from mart.company_weekly_kpis
    where (year = 2026 and week between 40 and 46) or (year = 2025 and week between 42 and 45)
    order by year, week
""").show(20, 0)

In [ ]:
import matplotlib.pyplot as plt

kpi = spark.table("mart.company_weekly_kpis").orderBy("week_start").toPandas()
fig, ax = plt.subplots(figsize=(12, 3.8))
ax.plot(kpi["week_start"], kpi["net_sales"].astype(float) / 1e5, color="#0f6e78", linewidth=1.6)
ax.set_ylabel("Net sales per week (₹ lakh)")
ax.set_title("Weekly net sales, Nov 2023 - Nov 2026: growth, Diwali peaks, Holi, monsoon dips")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 7. Data quality results (`qa` database)

The QA DAGs save every check to Hive. A company-level week-on-week change beyond ±20%, or a year-on-year change beyond ±40%, fails `fmcg_4_mart_qa`. Region and category breaches are recorded as warnings.

In [ ]:
spark.sql("""
    select check_name, level, status, count(*) as evaluations
    from qa.mart_checks
    group by check_name, level, status
    order by check_name, level, status
""").show(30, 0)

In [ ]:
# company-level weeks that breached a threshold, with the numbers behind them
spark.sql("""
    select check_name, concat(iso_year, '-W', lpad(iso_week, 2, '0')) as week, round(observed, 1) as change_pct, detail
    from qa.mart_checks
    where level = 'company' and status = 'FAIL'
    order by iso_year, iso_week, check_name
""").show(40, 0)

In [ ]:
spark.table("qa.datalake_checks").groupBy("check_name", "table_name", "status").count() \
     .orderBy("check_name", "table_name", "status").show(20, 0)

## 8. Browse the buckets with `gsutil` / `gcloud storage`

Shell commands work in cells with `!`. The kernel points the Google Cloud CLI at the local GCS emulator through a separate gcloud configuration (`fmcg-emulator`), so your default gcloud setup is untouched. With real GCS, the same commands run against your project.

In [3]:
!gsutil ls -lh gs://fmcg-datalake/transactions/year=2026/

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
                                  gs://fmcg-datalake/transactions/year=2026/month=1/
                                  gs://fmcg-datalake/transactions/year=2026/month=10/
                                  gs://fmcg-datalake/transactions/year=2026/month=11/
                                  gs://fmcg-datalake/transactions/year=2026/month=2/
                                  gs://fmcg-datalake/transactions/year=2026/month=3/
                                  gs://fmcg-datalake/transactions/year=2026/month=4/
                                  gs://fmcg-datalake/transactions/year=2026/month=5/
                                  gs://fmcg-datalake/transactions/year=2026/month=6/
                                  gs://fmcg-datalake/transa

In [ ]:
!gsutil du -sh gs://fmcg-landing gs://fmcg-datalake gs://fmcg-mart

In [ ]:
# the raw daily drops in the landing bucket
!gsutil ls -l gs://fmcg-landing/transactions/2026/11/08/ gs://fmcg-landing/customers/2026/11/08/

In [ ]:
# one datalake partition = one GCS folder of Parquet files
!gsutil ls -lh gs://fmcg-datalake/transactions/year=2026/month=11/day=8/

In [ ]:
!gcloud storage ls gs://fmcg-mart/fact_sales/

In [ ]:
# the reconciliation manifest written by the mart job
!gsutil cat gs://fmcg-mart/_manifests/2026/W45/manifest_20261108.json

In [ ]:
# Spark can read the raw .csv.gz straight from the landing bucket too
(spark.read.option("header", True)
      .csv("gs://fmcg-landing/transactions/2026/11/08/transactions_20261108.csv.gz")
      .select("transaction_id", "store_id", "product_name", "quantity", "unit_price", "payment_mode")
      .show(5, 0))